## 目标

本 Notebook 面向有后端/前端研发经验、但未系统学习深度学习的同学。

你不需要理解 BERT/Transformer 的数学细节；你只需要把下面这些跑通：

- 从 ModelScope 拉取数据集（`MsDataset`）
- 用预训练 backbone 做文本分类微调（`build_trainer`）
- 得到可用于本地推理的导出模型目录（`tmp/structbert_text_classification/output`）
- 用 `pipeline("text-classification")` 做推理


In [1]:
# 环境检查：确认当前工作目录与 Python 版本
import os
import sys

print("当前工作目录:", os.getcwd())
print("Python:", sys.version)

当前工作目录: /data/jzc/projects/bert-demo/bert_text_classification
Python: 3.11.15 (main, Mar 10 2026, 18:16:52) [Clang 21.1.4 ]


## 依赖安装

如果你还没安装依赖，在 Notebook 里可以直接执行下面的命令。

- 训练需要 `torch`、`modelscope` 等，建议使用 `requirements-all.txt`
- 如果你已经在环境里装过，可以跳过此步骤

## 补充：PyTorch / Miniconda / GPU（nvidia-smi）快速检查

这一小节用于你在“开始安装依赖/开训之前”快速确认环境是否正确，避免遇到 CUDA 不可用、驱动不匹配、占卡等常见问题。

### 1) Miniconda（推荐做隔离环境）

- 建议每个项目单独一个 Conda 环境，避免系统 Python 与依赖冲突。
- 常用命令：

```bash
# 查看 conda 版本与基础信息
conda --version
conda info

# 查看已有环境
conda env list

# 创建并进入环境（示例：Python 3.11）
conda create -n bert-demo python=3.11 -y
conda activate bert-demo

# 退出环境
conda deactivate
```

> 说明：如果你使用的是 `venv/poetry/uv` 等也没问题；核心是“隔离 + 可复现”。

### 2) PyTorch（确认是否识别到 CUDA）

- 推荐先在 Python 里做最小验证：

```python
import torch

print('torch:', torch.__version__)
print('cuda 可用:', torch.cuda.is_available())
```

- 常见现象与含义：
  - `cuda 可用: False`：通常是 **驱动未装好**、**容器/宿主未正确挂载 GPU**、或 **PyTorch 装成了 CPU 版本**。
  - 能看到设备名但训练仍慢：可能没把训练进程放到 GPU（例如 `device` 配置或环境变量不对）。

### 3) `nvidia-smi`（GPU 状态/占用/排障最常用）

- **看 GPU 基本信息与占用**：

```bash
nvidia-smi
```

- **持续刷新（每 1 秒）**：

```bash
watch -n 1 nvidia-smi
```

> 提示：如果 `nvidia-smi` 本身不可用，优先检查 NVIDIA 驱动是否正确安装、以及你当前是否在“可见 GPU”的环境（例如容器启动参数、集群资源分配）。


In [ ]:
# 前面的 "!" 符号是 Notebook 的魔法命令，表示在系统中执行下面的命令。
# 安装训练所需依赖（第一次运行可能需要几分钟）
!pip install -r ../requirements.txt

# 如果在 modelscope 的在线环境里，可以执行下面的命令，仅安装兼容的依赖版本
!pip install -r ../requirements/modelscope.txt

In [ ]:
# 导入依赖并打印关键版本，便于排障
import torch
import modelscope

print("torch:", torch.__version__)
print("cuda 可用:", torch.cuda.is_available())
print("cuda 设备数:", torch.cuda.device_count())
print("modelscope:", modelscope.__version__)

# 说明：没有 GPU 也可以先跑通逻辑，但训练会非常慢；建议至少 1 张 GPU

## 训练：加载数据集 + 构建 Trainer + 开始训练

这一段基本等价于 `bert_text_classification/train.py`，只是把关键变量写在一个 cell 里，方便你改参数后立即重跑。

> Bert 模型的结构图：
> 
> ![](https://github.com/datawhalechina/learn-nlp-with-transformers/blob/main/docs/%E7%AF%87%E7%AB%A02-Transformer%E7%9B%B8%E5%85%B3%E5%8E%9F%E7%90%86/pictures/3-bert-app.png?raw=true)

你需要关注 3 个东西：

- **数据集**：[DAMO_NLP/yf_dianping](https://www.modelscope.cn/datasets/DAMO_NLP/yf_dianping)
- **backbone**：[damo/nlp_structbert_backbone_base_std](https://www.modelscope.cn/models/iic/nlp_structbert_backbone_base_std)
- **输出目录**：`tmp/structbert_text_classification`

In [8]:
# 训练主流程（从 train.py 迁移而来，并补充了必要的中文说明）
import os

from modelscope import EpochBasedTrainer
from modelscope.metainfo import Trainers
from modelscope.msdatasets import MsDataset
from modelscope.trainers import build_trainer
from modelscope.utils.constant import ModelFile

# 训练输出目录：用于保存日志、checkpoint、以及最终导出的 output/ 模型目录
WORK_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "tmp", "structbert_text_classification"))

# 训练配置：直接复用仓库里的 configuration.json
CFG_FILE = os.path.abspath(os.path.join(os.getcwd(), ModelFile.CONFIGURATION))

# 说明：数据集字段名（文本列/标签列）。本示例的数据集就是 sentence/label
FIRST_SEQUENCE_KEY = "sentence"
LABEL_KEY = "label"

执行以上代码，导入相关的训练配置与依赖。以下训练代码建议直接运行本地的 `bert_text_classification/train.py` 文件。

In [ ]:
def build_cfg_modify_fn(train_ds: MsDataset):
    """返回 cfg_modify_fn：根据训练集规模，动态修正 LinearLR 的 total_iters。

    为什么需要它：
    - 配置文件里写死 total_iters 很容易和真实数据规模不一致
    - 这里用一个最简单的估算：steps_per_epoch * max_epochs
    """

    def cfg_modify_fn(cfg):
        # 只处理 LinearLR；如果你换了 scheduler，这段也不会影响训练
        if cfg.train.lr_scheduler.type == "LinearLR":
            bs = cfg.train.dataloader.batch_size_per_gpu
            steps_per_epoch = max(1, int(len(train_ds) / bs))
            cfg.train.lr_scheduler.total_iters = steps_per_epoch * cfg.train.max_epochs
        return cfg

    return cfg_modify_fn


# 1) 加载训练集与验证集（第一次会自动下载数据集到缓存）
train_dataset = MsDataset.load("DAMO_NLP/yf_dianping", split="train", subset_name="default")
eval_dataset = MsDataset.load("DAMO_NLP/yf_dianping", split="validation", subset_name="default")

print("训练集大小:", len(train_dataset))
print("验证集大小:", len(eval_dataset))

# 2) 构建 cfg_modify_fn
cfg_modify_fn = build_cfg_modify_fn(train_dataset)

# 3) 构建 Trainer
# 注意：model 传的是 ModelScope Hub 上的 backbone 名称
trainer: EpochBasedTrainer = build_trainer(
    name=Trainers.nlp_base_trainer,
    default_args=dict(
        model="damo/nlp_structbert_backbone_base_std",
        cfg_file=CFG_FILE,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        seed=42,
        work_dir=WORK_DIR,
        cfg_modify_fn=cfg_modify_fn,
        # Notebook 默认单进程单卡；多卡 DDP 建议用脚本 torchrun（见 README）
    ),
)

print("WORK_DIR:", WORK_DIR)
print("CFG_FILE:", CFG_FILE)

# 4) 开始训练
# 提示：如果你只想先验证流程，把 configuration.json 里的 max_epochs 改为 1
trainer.train()

## 训练产物检查

训练完成后，`WORK_DIR` 下通常会出现：

- 日志文件
- checkpoint
- `output/`：这是推理要用的**导出模型目录**

In [9]:
# 检查输出目录结构
import os

export_dir = os.path.join(WORK_DIR, "output")
print("导出模型目录:", export_dir)
print("是否存在:", os.path.isdir(export_dir))

if os.path.isdir(export_dir):
    # 只列出一层文件，避免输出太长
    print("output/ 下文件列表:")
    for name in sorted(os.listdir(export_dir)):
        print("-", name)
else:
    print("提示：如果 output/ 不存在，通常意味着训练尚未完成或中途失败，请先看上方训练日志。")

导出模型目录: /data/jzc/projects/bert-demo/tmp/structbert_text_classification/output
是否存在: True
output/ 下文件列表:
- README.md
- config.json
- configuration.json
- eval_result.txt
- pytorch_model.bin
- resources
- vocab.txt


## 推理：用导出模型目录做文本分类

这一步等价于 `bert_text_classification/inference.py`。

注意两点：

- `model=` 传的是**导出目录**（`.../output`），不是训练 work_dir 根目录
- `device`：有 GPU 用 `cuda`，否则可用 `cpu`（只是会慢一些）

In [10]:
# 推理示例：对一批中文评论打分
from modelscope.pipelines import pipeline

# 选择推理设备：优先 GPU
infer_device = "cuda" if torch.cuda.is_available() else "cpu"

remark_list = [
    "这家店的饭菜还不错的，值得再来",
    "这家店的饭菜一般的，不来也罢",
    "真是无语了，这家店",
    "位置偏僻，交通不便，但是还挺好吃的",
    "价格有点贵，但是量很足，味道不错",
    "服务态度很好，但是上菜速度有点慢",
    "环境不错，但是有点吵闹",
    "位置偏僻，服务很差，环境也一般，除了味道不错其他的都不行",
    "位置很近，服务很好，环境很好，价格也便宜，但是就菜不好吃",
]

# 构建文本分类 pipeline
# first_sequence 需要与 configuration.json 中的 preprocessor.first_sequence 一致
clf = pipeline(
    task="text-classification",
    model=export_dir,
    device=infer_device,
    first_sequence="sentence",
)

# topk=1：只取最高分的一个标签；batch_size 可按显存/内存调整
results = clf(remark_list, topk=1, batch_size=16)

for remark, result in zip(remark_list, results):
    # labels 里是字符串（例如 '0'/'1'），这里按 inference.py 的约定映射成“好/差”
    label = result["labels"][0]
    score = result["scores"][0]
    print(f"remark: {remark}\n  label: {label}（{'好' if label == '1' else '差'}）\n  score: {score:.4f}\n")

2026-03-31 17:16:02,580 - modelscope - WARNING - Use allow_remote=True. Will invoke codes from /data/jzc/projects/bert-demo/tmp/structbert_text_classification/output. Please make sure that you can trust the external codes.
2026-03-31 17:16:02,584 - modelscope - INFO - initiate model from /data/jzc/projects/bert-demo/tmp/structbert_text_classification/output
2026-03-31 17:16:02,584 - modelscope - INFO - initiate model from location /data/jzc/projects/bert-demo/tmp/structbert_text_classification/output.
2026-03-31 17:16:02,585 - modelscope - INFO - initialize model from /data/jzc/projects/bert-demo/tmp/structbert_text_classification/output
2026-03-31 17:16:03,034 - modelscope - INFO - The key of sentence1: sentence, The key of sentence2: None, The key of label: label
2026-03-31 17:16:03,038 - modelscope - INFO - The key of sentence1: sentence, The key of sentence2: None, The key of label: label


remark: 这家店的饭菜还不错的，值得再来
  label: 1（好）
  score: 0.9862

remark: 这家店的饭菜一般的，不来也罢
  label: 0（差）
  score: 0.9987

remark: 真是无语了，这家店
  label: 0（差）
  score: 0.9992

remark: 位置偏僻，交通不便，但是还挺好吃的
  label: 0（差）
  score: 0.9933

remark: 价格有点贵，但是量很足，味道不错
  label: 1（好）
  score: 0.9974

remark: 服务态度很好，但是上菜速度有点慢
  label: 1（好）
  score: 0.9775

remark: 环境不错，但是有点吵闹
  label: 1（好）
  score: 0.9879

remark: 位置偏僻，服务很差，环境也一般，除了味道不错其他的都不行
  label: 0（差）
  score: 0.9985

remark: 位置很近，服务很好，环境很好，价格也便宜，但是就菜不好吃
  label: 0（差）
  score: 0.9903



/data/jzc/projects/bert-demo/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
